# 03 - Netflix-paper reconstruction on MovieLens

## 1. Objective

Notebook 01 constructed the Netflix-paper visible matrix.
Notebook 02 computed the hidden activation probabilities using Eq. (2).
This notebook implements the reverse conditional distribution for visible reconstruction.

$$
p(v_i^k = 1 \mid h)
$$

This reconstructs rating probabilities for each movie.

## 2. Paper equations for reconstruction

### Eq. (1) Visible conditional distribution (from the paper)

$$
p(v_i^k = 1 \mid h) =
\frac{
\exp\left(b_i^k + \sum_{j=1}^{F} h_j W_{ij}^k\right)
}{
\sum_{l=1}^{K}
\exp\left(b_i^l + \sum_{j=1}^{F} h_j W_{ij}^l\right)
}
$$

Explanation of symbols:
- b_i^k is the visible bias for movie i at rating level k.
- The numerator gives the unnormalized score for rating level k.
- The denominator normalizes across all rating levels.
- Therefore this is a softmax over the K rating categories for each movie.

### Softmax form used in reconstruction

$$
softmax(z_k) =
\frac{\exp(z_k)}{\sum_l \exp(z_l)}
$$

$$
z_i^k =
 b_i^k + \sum_{j=1}^{F} h_j W_{ij}^k
$$

The visible reconstruction is obtained by applying a softmax across rating levels for each movie column.

## 3. Recreate the example user pipeline

This section rebuilds the same example user and visible matrix used in the previous notebooks.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path


def resolve_project_root() -> Path:
    root = Path.cwd().resolve()
    if root.name == "notebooks":
        root = root.parent
    return root


def load_ratings_table() -> tuple[pd.DataFrame, Path]:
    root = resolve_project_root()
    candidates = [
        root / "data" / "processed" / "train_ratings.csv",
        root / "data" / "processed" / "ratings.csv",
        root / "data" / "processed" / "rating.csv",
        root / "data" / "rating.csv",
        root / "data" / "ratings.csv",
    ]

    for path in candidates:
        if path.exists():
            df = pd.read_csv(path)
            if {"userId", "movieId", "rating"}.issubset(df.columns):
                return df[["userId", "movieId", "rating"]].copy(), path

    data_dir = root / "data"
    search_dirs = [data_dir / "processed", data_dir]
    for directory in search_dirs:
        if not directory.exists():
            continue
        for path in sorted(directory.glob("*.csv")):
            try:
                df = pd.read_csv(path)
            except Exception:
                continue
            if {"userId", "movieId", "rating"}.issubset(df.columns):
                return df[["userId", "movieId", "rating"]].copy(), path

    raise FileNotFoundError("No ratings table found with columns userId, movieId, rating.")


def build_visible_matrix_for_user(user_ratings_df: pd.DataFrame, rating_levels: list[float]):
    df = user_ratings_df[["movieId", "rating"]].copy()
    df = df.sort_values("movieId").reset_index(drop=True)

    ordered_movie_ids = df["movieId"].tolist()
    ordered_ratings = [round(float(r), 1) for r in df["rating"].tolist()]

    rating_to_index = {round(r, 1): idx for idx, r in enumerate(rating_levels)}
    K = len(rating_levels)
    m = len(df)

    V = np.zeros((K, m), dtype=int)
    for col, rating in enumerate(ordered_ratings):
        if rating not in rating_to_index:
            raise ValueError(f"Rating {rating} not in rating_levels vocabulary.")
        row = rating_to_index[rating]
        V[row, col] = 1

    return V, ordered_movie_ids, ordered_ratings, rating_to_index


rating_levels = [0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0]
K = len(rating_levels)

ratings_df, ratings_path = load_ratings_table()
user_counts = ratings_df.groupby("userId").size().reset_index(name="count")
eligible_users = user_counts[user_counts["count"] >= 20]
eligible_users = eligible_users.sort_values(["count", "userId"], ascending=[False, True])

if eligible_users.empty:
    raise ValueError("No user found with at least 20 ratings.")

selected_user_id = int(eligible_users.iloc[0]["userId"])
user_ratings = ratings_df[ratings_df["userId"] == selected_user_id].copy()

V, ordered_movie_ids, ordered_ratings, rating_to_index = build_visible_matrix_for_user(
    user_ratings, rating_levels
)

m = V.shape[1]

## 4. Recreate the hidden layer from Eq. (2)

### Eq. (2) Hidden conditional distribution (reused from Notebook 02)

$$
p(h_j = 1 \mid V) =
\sigma\left(
 b_j +
 \sum_{i=1}^{m}
 \sum_{k=1}^{K}
 v_i^k W_{ij}^k
\right)
$$

$$
\sigma(x) = \frac{1}{1 + e^{-x}}
$$

$$
a_j =
 b_j +
 \sum_{i=1}^{m}
 \sum_{k=1}^{K}
 v_i^k W_{ij}^k
$$

In [ ]:
F = 8
rng = np.random.default_rng(42)
W = rng.normal(loc=0.0, scale=0.01, size=(K, m, F))
b_h = np.zeros((F,), dtype=float)


def compute_hidden_preactivation(V: np.ndarray, W: np.ndarray, b_h: np.ndarray) -> np.ndarray:
    hidden_sum = (V[:, :, None] * W).sum(axis=(0, 1))
    return b_h + hidden_sum


def sigmoid(x: np.ndarray) -> np.ndarray:
    return 1.0 / (1.0 + np.exp(-x))


hidden_preactivation = compute_hidden_preactivation(V, W, b_h)
hidden_probs = sigmoid(hidden_preactivation)
h_sampled = rng.binomial(1, hidden_probs)

## 5. Initialize visible bias

At this stage the visible bias is only an initialization for demonstration; in a trained RBM it would be learned from data.

In [ ]:
b_v = np.zeros((K, m), dtype=float)

## 6. Compute reconstruction scores

$$
z_i^k =
 b_i^k + \sum_{j=1}^{F} h_j W_{ij}^k
$$

In [ ]:
recon_scores = b_v + np.tensordot(W, h_sampled, axes=([2], [0]))

## 7. Convert reconstruction scores to visible probabilities

Each column of the visible probability matrix is normalized across rating levels.

In [ ]:
def softmax_columns(x: np.ndarray) -> np.ndarray:
    x_max = x.max(axis=0, keepdims=True)
    exp_x = np.exp(x - x_max)
    return exp_x / exp_x.sum(axis=0, keepdims=True)


P_visible = softmax_columns(recon_scores)
print("Columns sum to 1:", np.allclose(P_visible.sum(axis=0), 1.0))

## 8. Display reconstructed rating distributions

In [ ]:
from IPython.display import display

print("Selected user:", selected_user_id)
print("K:", K)
print("m:", m)
print("F:", F)
print("V shape:", V.shape)
print("W shape:", W.shape)
print("Hidden probabilities shape:", hidden_probs.shape)
print("h_sampled shape:", h_sampled.shape)
print("b_v shape:", b_v.shape)
print("Reconstruction score shape:", recon_scores.shape)
print("P_visible shape:", P_visible.shape)

preview_n = min(5, m)
print("First few ordered_movie_ids:", ordered_movie_ids[:preview_n])
print("First few ordered_ratings:", ordered_ratings[:preview_n])
print("First few hidden probabilities:", hidden_probs[:min(5, F)])
print("First few sampled hidden states:", h_sampled[:min(5, F)])

preview_movies = min(3, m)
for idx in range(preview_movies):
    movie_id = ordered_movie_ids[idx]
    probs = P_visible[:, idx]
    df = pd.DataFrame(
        {
            "rating_level": rating_levels,
            "probability": probs,
        }
    )
    print(f"\nMovie {movie_id}")
    display(df)

## 9. Predicted ratings

$$
\hat r_i =
\sum_{k=1}^{K}
rating_k \cdot p(v_i^k \mid h)
$$

In [ ]:
rating_levels_array = np.array(rating_levels)
predicted_ratings = (rating_levels_array[:, None] * P_visible).sum(axis=0)

preview_n = min(5, m)
preview_df = pd.DataFrame(
    {
        "movieId": ordered_movie_ids[:preview_n],
        "observed_rating": ordered_ratings[:preview_n],
        "predicted_rating": predicted_ratings[:preview_n],
    }
)
preview_df

## 10. Interpretation

The hidden layer compresses the observed ratings into latent feature activations.
Eq. (1) maps those hidden activations back into a probability distribution over rating levels.
The expected reconstructed rating provides a real-valued summary of that distribution.
At this stage the parameters are still random, so the reconstructed ratings are illustrative rather than learned.

## 11. Pipeline summary

Step 1 — visible encoding

$$
ratings \rightarrow V
$$

Step 2 — hidden activation using Eq. (2)

$$
V \rightarrow p(h \mid V)
$$

Step 3 — visible reconstruction using Eq. (1)

$$
h \rightarrow p(V \mid h)
$$

The next notebook will introduce parameter learning so that the reconstruction becomes data-driven rather than random.